# Member 2 — Full Reproduction: Persian Text → Continuous Stress Regression

**Role:** NLP Expert. Fine-tune ParsBERT on raw Persian forum `content` only, predict the
continuous `stress` score, and hand leakage-safe OOF + validation/test fold-ensemble
predictions to Member C for the final fusion model.

## What problem does this notebook solve?

The project target is a **continuous stress score**, not a simple positive/negative
sentiment label. This is why Member 2 is formulated as a **regression** problem: a
prediction of 6.8 should be understood as much closer to 7.2 than to 2.0. The final
project may later convert the continuous score into ordered classes, but the text model
itself learns the continuous target.

Member 2 is deliberately **text-only**. This separation is scientifically useful because
it lets us answer a clean question: *how much can contextual Persian language alone
explain?* Metadata and handcrafted/tabular signals are left to Member 1. The final fusion
then combines two genuinely different information sources rather than allowing the
Transformer to silently consume the same tabular variables.

## Why ParsBERT?

`HooshvareLab/bert-fa-zwnj-base` is a Persian-specific BERT model. A Persian pretrained
model is preferable to training a Transformer from scratch because the labeled stress
dataset is too small to learn Persian syntax and semantics from zero. Pretraining gives
us a language representation learned from a much larger Persian corpus; fine-tuning then
adapts that representation to this project-specific regression target. The model is also
ZWNJ/half-space aware, which is relevant for Persian tokenization.

## Why five grouped models instead of one model?

The final Member C model is a **stacker / late-fusion model**. A stacker must not be
trained on base-model predictions that were produced by a model which already saw the
same training row. Therefore, each training row receives an **out-of-fold (OOF)**
prediction: the row is predicted by the fold model for which it was held out. Validation
and test rows are not used to train any fold model, so we average all five fold-model
predictions to reduce variance and create the same deterministic text signal for every
holdout row.

This notebook is the **single entry point** of the reproducible Member 2 project. It:
1. validates the frozen input/data-split contract,
2. prepares and normalizes Persian text,
3. verifies the grouped folds and leakage barriers,
4. loads (or retrains) five fold-specific ParsBERT regressors,
5. generates deterministic OOF predictions for every training row,
6. generates five-model ensemble predictions for validation and test,
7. evaluates regression + ordered-class behavior,
8. performs post-hoc error analysis / token attribution,
9. exports the exact scalar predictions needed by Member C, and
10. verifies artifacts, tests, and reproducibility checks.

**Two execution modes:**
- `FULL_RETRAIN = False` (default): load the supplied fine-tuned fold checkpoints from `models/`.
- `FULL_RETRAIN = True`: retrain all five fold models from the accepted data (GPU strongly recommended).

> **Important methodological boundary:** `stress_proxy`, anxiety/depression annotation
> fields, metadata, and final-test information are not inputs to this text model. The
> model sees Persian post text and the supervised stress target during training only.


## 1. Environment and project setup

### What this block does

This block resolves the project root, imports the core scientific libraries, prints the
software versions, loads the frozen configuration files, sets random seeds, creates the
output directories, chooses CPU/GPU, and selects whether the notebook should retrain or
reuse the supplied checkpoints.

### Why these libraries?

- **NumPy** is used for numeric arrays and deterministic numerical operations.
- **pandas** is used because the project data are naturally tabular: one row per post with
  IDs, split roles, targets, weights, and predictions.
- **PyTorch** is the deep-learning engine used by Hugging Face Transformers. It performs
  gradient computation and GPU training/inference.
- **Hugging Face Transformers** provides the pretrained ParsBERT architecture/tokenizer
  and a standard interface for loading/saving fine-tuned checkpoints.
- **pathlib** is used instead of hard-coded operating-system paths so the project is more
  portable between Windows/Linux/Colab environments.

### Why print versions and set seeds?

Deep-learning results can change when package versions, tokenizers, or random
initialization change. Recording the environment makes a run auditable. Setting the same
seed reduces avoidable randomness in shuffling/model initialization. It does **not** make
every GPU kernel mathematically bit-identical across every hardware/software stack, but
it is an important reproducibility control.

### Why support both CPU and GPU?

Transformer training is computationally expensive. GPU execution is the intended route
for retraining. CPU execution is useful for lightweight validation or, when frozen
predictions are already available, for reproducing the downstream evaluation without
spending hours recomputing the neural network.


In [1]:
import sys, os, json, platform
from pathlib import Path

import numpy as np
import pandas as pd

# Resolve project root (the folder containing configs/ and src/)
PROJECT_ROOT = Path(os.getcwd()).resolve()
# fall back to notebook location if cwd is notebooks/
if not (PROJECT_ROOT / "configs").is_dir():
    PROJECT_ROOT = Path("..").resolve()
assert (PROJECT_ROOT / "configs").is_dir(), "project root not found"
sys.path.insert(0, str(PROJECT_ROOT))

from src.common import resolve_root, set_seed, load_project_config, load_model_config

PROJECT_ROOT = resolve_root(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)

import torch
from transformers import __version__ as tf_version

print("Python:", platform.python_version())
print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())
print("transformers:", tf_version)
print("numpy:", np.__version__, "| pandas:", pd.__version__)

cfg = load_project_config(PROJECT_ROOT)
mcfg = load_model_config(PROJECT_ROOT)
seed = cfg["random_seed"]
set_seed(seed)
print("seed:", seed)

for rel in cfg["output_paths"].values():
    (PROJECT_ROOT / rel).mkdir(parents=True, exist_ok=True)
print("output dirs created")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE =", DEVICE)

# Modes
FULL_RETRAIN = False   # set True to retrain fold models from scratch
print("FULL_RETRAIN =", FULL_RETRAIN)

Project root: C:\Users\Asus\Desktop\datasience\final_report\Member2_Final_Reproducible_Project


Python: 3.12.4
torch: 2.12.0+cpu | cuda available: False
transformers: 5.14.1
numpy: 2.4.6 | pandas: 2.3.3
seed: 42
output dirs created
DEVICE = cpu
FULL_RETRAIN = False


## 2. Input validation

### What this block does

Before any model work, the notebook loads the Member 2 handoff and the global modeling
manifest, joins them by `unique_post_id`, normalizes basic dtypes, and runs the frozen
input-contract checks.

### Why validate *before* preprocessing or training?

A model can produce plausible metrics even when the data contract is wrong. Silent
problems such as duplicated IDs, missing rows, incorrect split roles, or an invalid fold
assignment are more dangerous than a visible runtime error because they can create
**data leakage**. We therefore fail early.

The important invariants are:

1. every post has a unique stable join key,
2. train / validation / test / embargo roles are mutually exclusive,
3. only training rows have OOF fold assignments,
4. the target lies on the expected stress scale,
5. the frozen role counts/fold structure are consistent with Member C's manifest, and
6. no excluded/embargo row is accidentally used for model fitting or selection.

### Why merge with the modeling manifest?

The Member 2 file contains the text-model fields, while the global manifest contains the
leakage-control metadata (`group_id`, author, thread, content hash, and role flags). The
join lets the text notebook independently verify that it is respecting the project-wide
split rather than trusting a local CSV blindly.


In [2]:
from src.validate_inputs import validate_inputs, check_no_overlap

handoff = pd.read_csv(PROJECT_ROOT / "inputs" / "member2_handoff.csv")
manifest = pd.read_csv(PROJECT_ROOT / "inputs" / "modeling_manifest_v2.csv")

df = handoff.merge(
    manifest[["unique_post_id","group_id","author","thread_id","content_hash",
              "official_eval","use_for_training","use_for_model_selection",
              "use_for_final_test","exclude_from_modeling"]],
    on="unique_post_id", how="left",
)
df["content"] = df["content"].fillna("").astype(str)
df["oof_fold"] = df["oof_fold"].astype("Int64")

summary = validate_inputs(df, cfg)
print("Contract validation PASSED")
print("  total rows:", summary["total_rows"])
print("  train:", summary["train"], "| validation:", summary["validation"],
      "| test:", summary["test"], "| embargo:", summary["embargo"])
print("  fold sizes:", summary["fold_sizes"])
print("  unique ids:", summary["unique_ids"])

Contract validation PASSED
  total rows: 5615
  train: 4226 | validation: 452 | test: 453 | embargo: 484
  fold sizes: {0: 844, 1: 844, 2: 848, 3: 847, 4: 843}
  unique ids: 5615


## 3. Data preparation

### What this block does

The notebook separates the frozen modeling roles and creates a normalized Persian text
column (`content_norm`). The **raw `content` remains the conceptual source**, while the
normalized version is what is passed to the Transformer tokenizer.

### Why normalize Persian text?

Persian user-generated text contains orthographic variation: Arabic/Persian variants of
letters such as yeh/kaf, inconsistent whitespace, and ZWNJ/half-space patterns. These
variations can cause two visually similar words to be split into different token
sequences. `hazm.Normalizer` reduces this superficial variation before tokenization.

This is different from the early corpus-cleaning stage, where raw text was preserved for
surface features such as punctuation and emoji counts. Member 2 does not use those raw
surface features; it needs a stable textual representation for ParsBERT.

### Why is only `content` used as model input?

This is a deliberate **modality-isolation** decision. If Member 2 also used category,
gender, lexical counts, `stress_proxy`, or other engineered fields, its prediction would
no longer be a clean text-only signal. Member C's late fusion would then combine highly
overlapping models and the interpretation of complementarity would be weaker.

`final_stress` is the supervised regression target. `training_sample_weight` is not a
predictor; it only changes how much each training example contributes to the loss. The
`clinical_class` and fold columns are used for evaluation/splitting, not as text-model
features.

### Why save a processed manifest?

A reproducible pipeline should record *what it actually processed*. The manifest captures
row counts, modeling roles, columns used, and the normalization method so that a later
run can be compared with the original one without inspecting notebook state manually.


In [3]:
from src.prepare_data import make_normalizer, clean_persian_text

normalizer = make_normalizer()
for role in df["model_role"].unique():
    m = df["model_role"] == role
    print(f"{role:10s}: {int(m.sum())} rows")

df["content_norm"] = df["content"].map(lambda t: clean_persian_text(t, normalizer))

train_df = df[df["model_role"] == "train"].reset_index(drop=True)
val_df   = df[df["model_role"] == "validation"].reset_index(drop=True)
test_df  = df[df["model_role"] == "test"].reset_index(drop=True)
embargo_df = df[df["model_role"] == "embargo"]

print("train/val/test/embargo prepared")
print("sample cleaned text:")
print(df[["content","content_norm"]].iloc[0].to_dict())

# Save processed data manifest
import json
proc_manifest = {
    "rows_processed": int(len(df)),
    "train": int(len(train_df)), "validation": int(len(val_df)),
    "test": int(len(test_df)), "embargo": int(len(embargo_df)),
    "columns_used": ["content", "final_stress", "training_sample_weight", "clinical_class", "oof_fold"],
    "text_cleaning": "hazm.Normalizer",
}
with open(PROJECT_ROOT / "data" / "processed" / "processed_manifest.json", "w", encoding="utf-8") as f:
    json.dump(proc_manifest, f, indent=2, ensure_ascii=False)
print("processed manifest saved")

train     : 4226 rows
validation: 452 rows
test      : 453 rows
embargo   : 484 rows


train/val/test/embargo prepared
sample cleaned text:
{'content': 'قد و وزنشو بگو و چندسالشع؟', 'content_norm': 'قد و وزنشو بگو و چندسالشع؟'}
processed manifest saved


## 4. Grouped fold handling

### What this block does

The notebook loads the exact five training folds supplied by Member C and verifies that
the project-level leakage groups do not cross fold boundaries. It also checks that the
four modeling roles have disjoint IDs.

### Why ordinary random K-fold CV would be unsafe here

Forum rows are not independent. Posts can share:

- the same **author** (writing style and personal context),
- the same **thread** (shared conversation/topic context), and
- the same or duplicated **content**.

If related rows were placed in both the training and held-out fold, the model could look
better simply because it recognizes the same author/thread/text pattern. Grouped folds
force connected examples to move together and therefore give a more realistic estimate
of generalization.

### Why five folds?

Five folds provide a practical compromise. Each fold model trains on about 80% of the
training data and predicts the remaining ~20%. This produces one honest OOF prediction
for every training row while keeping the number of expensive Transformer training runs
manageable. More folds would reduce the held-out fraction but substantially increase GPU
cost; fewer folds would train each model on less data and give a noisier stacker input.

### Why verify role separation again?

Defense-in-depth is intentional. The input validator checks the contract globally; this
block checks it again at the point where fold logic is used. Repeated invariants are cheap
compared with the cost of discovering leakage after model training.


In [4]:
from src.build_folds import load_folds, check_grouped_no_leak

folds = load_folds(df, cfg)
for f, ids in folds.items():
    print(f"fold {f}: {len(ids)} train rows")

print()
print("grouped leak check (author/thread/content_hash crossing folds):")
leak_report = check_grouped_no_leak(df, cfg)
for k, v in leak_report.items():
    print(f"  {k}: {v}")

# role separation (no shared IDs)
check_no_overlap(df, cfg)
print("role separation OK (no shared IDs between train/val/test/embargo)")

fold 0: 844 train rows
fold 1: 844 train rows
fold 2: 848 train rows
fold 3: 847 train rows
fold 4: 843 train rows

grouped leak check (author/thread/content_hash crossing folds):
  author: {'groups_crossing_folds': 0, 'groups_total': 3686}
  thread_id: {'groups_crossing_folds': 0, 'groups_total': 3655}
  content_hash: {'groups_crossing_folds': 0, 'groups_total': 3739}
role separation OK (no shared IDs between train/val/test/embargo)


## 5. Model training (ParsBERT regression)

Base model: `HooshvareLab/bert-fa-zwnj-base` (ParsBERT v3, ZWNJ-aware), configured as
`BertForSequenceClassification(num_labels=1, problem_type="regression")`.

### Why a pretrained Transformer?

BERT-style self-attention can represent a word differently depending on its surrounding
words. This matters for stress language because the meaning of a phrase is often
contextual; simple counts cannot reliably distinguish negation, reassurance, uncertainty,
or a stressful narrative. Starting from Persian pretraining is data-efficient: the
network already knows general Persian language structure and only needs to learn the
stress-related mapping during fine-tuning.

### Why a single regression output?

`num_labels=1` gives one continuous score. This matches the continuous project target and
preserves ordering/severity information. The four project classes are evaluation/
operational bins applied later; they are not what the neural network directly optimizes.

### Tokenizer and `max_length`

The tokenizer converts normalized Persian text into subword IDs that ParsBERT understands.
The notebook measures token-length statistics before relying on the configured
`max_length`. This is preferable to choosing a length arbitrarily: too short loses useful
context through truncation, while too long wastes GPU memory and slows every batch. The
final configuration uses 384 tokens, which covers almost all posts while accepting that a
small long-tail of extremely long posts must be truncated.

### Training loss: weighted asymmetric MSE

For a prediction \(\hat y_i\), target \(y_i\), and training weight \(w_i\), the project
uses the idea

\[
L_i = w_i \; d_i \; (\hat y_i-y_i)^2,
\]

where \(d_i=1.75\) when the model **under-predicts** (`prediction < target`) and 1 otherwise.

There are two reasons:

1. **Sample weighting:** the label distribution is imbalanced and annotation sources have
   different confidence/provenance. `training_sample_weight` prevents the large/easy
   portion of the data from completely dominating optimization.
2. **Asymmetric error cost:** within this project's safety-oriented objective,
   underestimating a truly high-stress post is considered more costly than an equally
   large overestimate. The factor 1.75 introduces a conservative bias. It is a project
   design choice, **not a clinically validated universal constant**.

Squared error is still useful because larger mistakes receive more penalty than small
mistakes. The asymmetry modifies the direction-specific cost without changing the basic
regression objective.

### Why train five separate fold models?

For fold `f`, the model trains on the other four folds and uses fold `f` as its held-out
fold. This creates the exact models needed for leakage-safe OOF stacking. The notebook
uses early stopping so training ends when held-out MAE no longer improves, reducing both
overfitting and unnecessary GPU work.

When `FULL_RETRAIN=False`, the notebook loads the already accepted fold checkpoints. This
is the normal reproducibility path: we verify/evaluate the frozen model rather than
randomly changing it through a new training run.


In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(mcfg["model_name"])
print("tokenizer:", mcfg["model_name"])
print("max_length:", mcfg["max_length"])

from src.prepare_data import token_length_stats
stats = token_length_stats(train_df["content_norm"], tokenizer)
print("train token-length percentiles:", {k: round(v,1) for k,v in stats.items()})

tokenizer: HooshvareLab/bert-fa-zwnj-base
max_length: 384


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (637 > 512). Running this sequence through the model will result in indexing errors


train token-length percentiles: {50: 40.0, 90: 110.0, 95: 151.8, 99: 262.0, 100: 1071.0}


In [6]:
from src.train_models import train_fold
import torch

os.environ["TOKENIZERS_PARALLELISM"] = "false"

if FULL_RETRAIN:
    print("FULL_RETRAIN mode: training 5 fold models from scratch")
    fold_summaries = {}
    for f in range(cfg["num_folds"]):
        tr = train_df[train_df["oof_fold"] != f].reset_index(drop=True)
        va = train_df[train_df["oof_fold"] == f].reset_index(drop=True)
        print(f"--- fold {f}: train {len(tr)} held-out {len(va)} ---")
        res = train_fold(
            tr["content_norm"].tolist(), tr["final_stress"].values,
            tr["training_sample_weight"].values,
            va["content_norm"].tolist(), va["final_stress"].values,
            mcfg["model_name"], tokenizer,
            mcfg["max_length"], mcfg["batch_size"],
            mcfg["learning_rate"], mcfg["weight_decay"],
            mcfg["fold_epochs"], mcfg["early_stopping_patience"],
            mcfg["loss_params"]["under_prediction_penalty"], seed, DEVICE,
            PROJECT_ROOT / "models" / f"fold_{f}",
        )
        fold_summaries[f] = res
        print("   best val_mae:", round(res["best_val_mae"],4))
    with open(PROJECT_ROOT / "data" / "processed" / "fold_training_summary.json","w") as fh:
        json.dump(fold_summaries, fh, indent=2)
else:
    print("Loading supplied fold checkpoints from models/ ...")
    from src.verify_artifacts import verify_models
    verify_models(PROJECT_ROOT / "models", cfg["num_folds"])
    print("All 5 fold checkpoints present and verified")

Loading supplied fold checkpoints from models/ ...
All 5 fold checkpoints present and verified


## 6. OOF prediction generation (deterministic)

### What is an OOF prediction?

For every training row, the notebook uses **only the model whose held-out fold contains
that row**. The model therefore did not optimize its weights on that example. Concatenating
all five held-out predictions gives exactly one leakage-safe text prediction for every
training post.

This is essential for Member C's stacker. If we instead gave Member C in-sample training
predictions, the text model would appear unrealistically accurate on the stacker's
training data. The fusion model could learn to trust a signal that will be noisier at
validation/test/deployment time — a classic stacking leakage problem.

`model.eval()` disables training-time dropout behavior. We also do not use MC-dropout in
the primary prediction contract. The goal here is a **deterministic scalar feature** that
can be regenerated and compared across runs.

## 7. Validation and test prediction generation (five-fold ensemble)

Validation and test rows were not used to fit any of the five fold models. Therefore,
each of the five models can predict each holdout row honestly. The final Member 2 signal
is the arithmetic mean of the five predictions.

### Why average the five models?

Individual fold models see slightly different training subsets and therefore have model
variance. Averaging reduces dependence on any one fold and generally produces a more
stable estimate. It also creates one consistent scalar input for the final fusion model.

### Why is the test not used for selection?

The test set is evidence, not a tuning tool. Model architecture, fold protocol, training
settings, and downstream fusion choices must be fixed before test evaluation. Repeatedly
changing the model after seeing test performance would turn the test into another
validation set and make the reported generalization estimate optimistic.

### Prediction backends

- `"modal"` — run the five supplied checkpoints on a Modal GPU. This is fast and performs
  actual neural-network inference.
- `"cpu"` — perform the same inference locally without a GPU. Correct but much slower.
- `"load"` — load the already-generated deterministic handoff predictions. This is useful
  when large checkpoints/GPU access are unavailable and the goal is to reproduce metrics,
  contracts, and Member C integration rather than recompute the neural network.

The backend changes **where/how inference is executed**, not the scientific definition of
OOF and fold-ensemble predictions.


In [7]:
from src.generate_predictions import generate_all_predictions, pick_backend
from src.modal_generate import modal_available

# Auto-select backend; override with PREDICT_BACKEND below if desired.
PREDICT_BACKEND = None  # auto: modal -> load -> cpu  # e.g. "modal", "cpu", "load"
handoff_dir = PROJECT_ROOT / "data" / "handoff"
handoff_dir.mkdir(parents=True, exist_ok=True)
# copy the frozen inputs into data/handoff if not already present
import shutil
for fname in ["member2_handoff.csv", "modeling_manifest_v2.csv"]:
    srcp = PROJECT_ROOT / "inputs" / fname
    dstp = handoff_dir / fname
    if not dstp.exists():
        shutil.copy2(srcp, dstp)

regenerated_exist = (
    (handoff_dir / "member2_oof_predictions_deterministic.csv").exists()
    and (handoff_dir / "member2_validation_predictions_fold_ensemble.csv").exists()
    and (handoff_dir / "member2_test_predictions_fold_ensemble.csv").exists()
)
backend = pick_backend(modal_available(), regenerated_exist, PREDICT_BACKEND)
print("PREDICTION BACKEND:", backend)

preds = generate_all_predictions(
    df, cfg, mcfg, PROJECT_ROOT / "models", backend, DEVICE, handoff_dir, PROJECT_ROOT,
)
oof_pred = preds["oof"]
val_pred = preds["validation"]
test_pred = preds["test"]

# Save to outputs/predictions (canonical handoff location)
pred_out = PROJECT_ROOT / "outputs" / "predictions"
pred_out.mkdir(parents=True, exist_ok=True)
oof_pred.to_csv(pred_out / "member2_oof_predictions_deterministic.csv", index=False)
val_pred.to_csv(pred_out / "member2_validation_predictions_fold_ensemble.csv", index=False)
test_pred.to_csv(pred_out / "member2_test_predictions_fold_ensemble.csv", index=False)

print("OOF rows:", len(oof_pred), "| fold dist:", oof_pred["fold"].value_counts().sort_index().to_dict())
print("OOF MAE:", round(np.abs(oof_pred["prediction"]-oof_pred["true_stress"]).mean(),4))
print("validation rows:", len(val_pred), "| MAE:", round(np.abs(val_pred["prediction"]-val_pred["true_stress"]).mean(),4))
print("test rows:", len(test_pred), "| MAE:", round(np.abs(test_pred["prediction"]-test_pred["true_stress"]).mean(),4))
print(val_pred.head())

PREDICTION BACKEND: modal


volume already has all required data (no upload needed)
volume sync complete


OOF rows: 4226 | fold dist: {0: 844, 1: 844, 2: 848, 3: 847, 4: 843}
OOF MAE: 0.9845
validation rows: 452 | MAE: 0.8246
test rows: 453 | MAE: 0.9689
  unique_post_id  model_role  true_stress  prediction  \
0      253589219  validation     3.740741    4.821971   
1      253589632  validation     2.000000    4.210680   
2      253590070  validation     1.370370    1.236816   
3      253590597  validation     1.000000    2.207247   
4      256431909  validation     1.000000    1.294978   

   prediction_std_fold_models  fold_model_0  fold_model_1  fold_model_2  \
0                    0.345457      4.956089      4.325316      4.829472   
1                    0.626403      4.455866      5.013816      4.530651   
2                    0.180948      0.923320      1.371697      1.384508   
3                    0.403516      2.436822      1.976372      2.690722   
4                    0.090179      1.196326      1.312758      1.255780   

   fold_model_3  fold_model_4  
0      5.366210      4.63

## 8. Evaluation

### Why use both regression and class-based metrics?

The model is trained as a regressor, so continuous metrics tell us how close predicted
stress values are to the annotated values. The project also has ordered operational
classes, so class metrics tell us whether errors cross important severity boundaries.
Neither view is sufficient by itself.

### Continuous metrics

- **MAE (Mean Absolute Error):** average absolute distance between prediction and target.
  It is easy to interpret on the original 1–10 stress scale and is the main continuous
  metric used for model comparison.
- **RMSE:** square root of mean squared error. It penalizes large mistakes more strongly
  than MAE, so it is useful for detecting occasional severe errors.
- **Pearson correlation:** measures linear association between true and predicted scores.
  A high value means the model tracks relative severity well, but correlation alone does
  not guarantee accurate calibration.
- **Spearman correlation:** evaluates monotonic/rank ordering and is less tied to a
  strictly linear relationship.

### Ordered-class metrics

The authoritative classes are **Low ≤ 3**, **Moderate (3,5]**, **High (5,7]**, and
**Very High > 7**. Accuracy is reported, but the class distribution is imbalanced, so we
also report per-class precision/recall/F1 and **macro F1**, which gives each class equal
weight regardless of support.

Recall is particularly important for the high-risk classes because it answers: *of the
posts that truly belong to this class, how many did we recover?* Precision answers a
different question: *of the posts we predicted as this class, how many were actually in
it?* The two must not be confused.

### Why confusion matrices and true-vs-predicted scatter plots?

A single metric hides the shape of errors. The confusion matrix shows **where** a class is
being confused (e.g., Moderate → Low versus Moderate → High). The scatter plot shows
continuous behavior, including regression-to-the-mean and whether predictions preserve
the ordering of stress severity.

OOF, validation, and test are printed separately because they answer different questions:
OOF describes training-set generalization under the fold protocol; validation is used for
development/model selection; the locked test is the final untouched evidence.


In [8]:
from src.evaluate_model import evaluate_predictions, save_metrics, plot_confusion_matrix

oof_pred_c = oof_pred.copy()
oof_pred_c["clinical_class"] = train_df.set_index("unique_post_id").loc[oof_pred_c["unique_post_id"], "clinical_class"].values

val_pred_c = val_pred.copy()
val_pred_c["clinical_class"] = val_df.set_index("unique_post_id").loc[val_pred_c["unique_post_id"], "clinical_class"].values

test_pred_c = test_pred.copy()
test_pred_c["clinical_class"] = test_df.set_index("unique_post_id").loc[test_pred_c["unique_post_id"], "clinical_class"].values

oof_rep = evaluate_predictions(oof_pred_c, cfg, "oof")
val_rep = evaluate_predictions(val_pred_c, cfg, "validation")
test_rep = evaluate_predictions(test_pred_c, cfg, "test")

for rep in (oof_rep, val_rep, test_rep):
    print(rep["split"], "| MAE", round(rep["MAE"],4), "| RMSE", round(rep["RMSE"],4),
          "| Pearson", round(rep["Pearson"],4), "| Spearman", round(rep["Spearman"],4),
          "| acc", round(rep["accuracy"],4), "| macroF1", round(rep["macro_f1"],4))

save_metrics(oof_rep, PROJECT_ROOT/"outputs"/"metrics"/"oof_metrics.json")
save_metrics(val_rep, PROJECT_ROOT/"outputs"/"metrics"/"validation_metrics.json")
save_metrics(test_rep, PROJECT_ROOT/"outputs"/"metrics"/"test_metrics.json")
plot_confusion_matrix(val_rep, PROJECT_ROOT/"outputs"/"figures"/"confusion_matrix_validation.png")
plot_confusion_matrix(test_rep, PROJECT_ROOT/"outputs"/"figures"/"confusion_matrix_test.png")
print("metrics + figures saved")

oof | MAE 0.9845 | RMSE 1.3193 | Pearson 0.813 | Spearman 0.8003 | acc 0.6488 | macroF1 0.5571
validation | MAE 0.8246 | RMSE 1.1462 | Pearson 0.8596 | Spearman 0.8061 | acc 0.7412 | macroF1 0.6224
test | MAE 0.9689 | RMSE 1.3182 | Pearson 0.8251 | Spearman 0.7891 | acc 0.7285 | macroF1 0.619


metrics + figures saved


In [9]:
import matplotlib.pyplot as plt

for title, pred_df in [("Validation", val_pred_c), ("Test", test_pred_c)]:
    plt.figure(figsize=(6,6))
    plt.scatter(pred_df["true_stress"], pred_df["prediction"], alpha=0.5, s=16)
    plt.plot([1,10],[1,10],"r--")
    plt.xlabel("true stress"); plt.ylabel("predicted stress")
    plt.title(f"{title}: predicted vs true")
    plt.xlim(1,10); plt.ylim(1,10)
    plt.tight_layout()
    split = title.lower()
    plt.savefig(PROJECT_ROOT/"outputs"/"figures"/f"scatter_{split}.png", dpi=110)
    plt.show()
print("scatter plots saved")

C:\Users\Asus\AppData\Local\Temp\ipykernel_22208\1540752424.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


scatter plots saved


## 9. Explainability

### What explainability means here

The production text pipeline did **not** use SHAP, attention scores, Integrated Gradients,
or token attribution to choose features or tune the model. Explainability is performed
**after training** to understand behavior and errors.

This distinction matters scientifically: an explanation method should not be presented as
part of the predictive algorithm unless it actually influenced training. Here it is an
audit/reporting layer.

### 1) Error analysis

The notebook separates under-prediction and over-prediction and summarizes error by stress
band. This is more informative than MAE alone because the custom loss intentionally treats
under-prediction as more costly. We therefore check whether the trained model actually
shows the expected directional behavior and where the largest errors occur.

### 2) Simple gradient-based token attribution

For a small number of validation examples, the notebook estimates which input tokens have
the largest local gradient influence on the predicted score. Intuitively, the gradient
asks: *if the representation of this token changed slightly, how sensitive would the model
output be?*

This is only a **local approximation**. It does not prove that a token causes stress, and
Transformer explanations can be unstable because contextual representations interact.
The output is therefore appropriate for qualitative inspection/reporting, not for clinical
interpretation or automatic feature selection.


In [10]:
from src.explain_model import run_error_analysis, save_explainability, example_token_attribution

err_rep = run_error_analysis(val_pred_c, cfg)
print("validation error analysis:")
print("  under:", round(err_rep["pct_under_total"],1), "% | over:", round(err_rep["pct_over_total"],1), "%")
for band, d in err_rep["band_error"].items():
    print(f"  {band}: n={d['n']} mean_abs_err={d['mean_abs_error']:.3f} pct_under={d['pct_under']:.1f}%")

save_explainability(err_rep, PROJECT_ROOT/"outputs"/"explainability"/"error_analysis_validation.json")
print("error analysis saved")

validation error analysis:
  under: 36.1 % | over: 63.9 %
  Low: n=297 mean_abs_err=0.742 pct_under=24.2%
  Moderate: n=67 mean_abs_err=1.021 pct_under=44.8%
  High: n=63 mean_abs_err=0.983 pct_under=68.3%
  Very High: n=25 mean_abs_err=0.880 pct_under=72.0%
error analysis saved


In [11]:
# Token attribution on 2 example posts (interpretation only)
from transformers import AutoModelForSequenceClassification

m0 = AutoModelForSequenceClassification.from_pretrained(str(PROJECT_ROOT/"models"/"fold_0"))
m0.eval()
if DEVICE == "cuda":
    m0 = m0.cuda()

examples = df[df["model_role"]=="validation"].head(2)["content_norm"].tolist()
explanations = {}
for i, text in enumerate(examples):
    attrs = example_token_attribution(text, m0, tokenizer, mcfg["max_length"], DEVICE)
    top = attrs[:8]
    explanations[f"example_{i}"] = {"text": text, "top_tokens": [(t, round(a,3)) for t,a in top]}
    print(f"example {i} top tokens:", [(t, round(a,3)) for t,a in top])

save_explainability(explanations, PROJECT_ROOT/"outputs"/"explainability"/"token_attribution_examples.json")
print("token attribution saved (interpretation only)")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

example 0 top tokens: [('استرس', -3.619), ('دکتر', -1.832), ('یادم', -1.714), ('رفت', -1.232), ('بیمه', -1.183), ('بیمه', -1.031), ('##زاد', -0.957), ('گفت', -0.8)]


example 1 top tokens: [('فشار', -3.801), ('نه', -3.12), ('اینا', 1.624), ('زده', 1.295), ('سندروم', -1.003), ('داره', 0.984), ('برام', 0.884), ('خودمم', 0.858)]
token attribution saved (interpretation only)


## 10. Member C handoff export

### What is exported?

The final fusion does not need Member 2's training code internally. It needs a clean,
leakage-safe scalar prediction keyed by `unique_post_id`:

- one OOF prediction for every training row,
- one five-fold ensemble prediction for every validation row,
- one five-fold ensemble prediction for every test row.

### Why scalar predictions rather than raw 768-D embeddings in the frozen final fusion?

A scalar prediction is compact, directly optimized for the stress target, and has a clear
OOF definition. Earlier exploratory work considered Transformer embeddings, but the frozen
primary Member C contract uses the scalar Member 2 prediction. This avoids a representation
mismatch between fold-specific OOF encoders and a separate full-train encoder and makes
the final stacker easier to audit and reproduce.

### Why key everything by `unique_post_id`?

Row order can change after filtering, saving, or merging. Joining by a stable unique key
prevents accidental row-wise misalignment between Member 1, Member 2, labels, and the
fusion model. The export therefore treats IDs as part of the scientific contract, not as
incidental bookkeeping.


In [12]:
from src.export_memberC_handoff import export_handoff

handoff_result = export_handoff(
    oof_pred, val_pred, test_pred, cfg,
    PROJECT_ROOT / "outputs" / "predictions",
)
print(json.dumps(handoff_result, indent=2, ensure_ascii=False))

{
  "oof_path": "C:\\Users\\Asus\\Desktop\\datasience\\final_report\\Member2_Final_Reproducible_Project\\outputs\\predictions\\member2_oof_predictions_deterministic.csv",
  "validation_path": "C:\\Users\\Asus\\Desktop\\datasience\\final_report\\Member2_Final_Reproducible_Project\\outputs\\predictions\\member2_validation_predictions_fold_ensemble.csv",
  "test_path": "C:\\Users\\Asus\\Desktop\\datasience\\final_report\\Member2_Final_Reproducible_Project\\outputs\\predictions\\member2_test_predictions_fold_ensemble.csv",
  "oof_rows": 4226,
  "val_rows": 452,
  "test_rows": 453
}


## 11. Artifact verification

### Why verify artifacts after computing metrics?

Reproducibility is not only "the code runs." A complete reproducible handoff must prove
that the *right rows* were predicted by the *right model family* and that the expected
files can be reused later.

This block verifies:

- the five fold-model directories/checkpoints are available for full inference,
- OOF predictions cover every training ID exactly once,
- validation/test predictions cover their complete frozen roles,
- embargo IDs are absent,
- no duplicate prediction IDs exist,
- metric and figure files are present, and
- the expected prediction files are written.

### Why write SHA-256 checksums?

A checksum is a content fingerprint. If a CSV changes by even one byte, its SHA-256 hash
changes. Storing hashes makes it possible to detect accidental edits/corruption and to
confirm that Member C received the same frozen prediction artifacts that were evaluated
here. Checksums do not prove scientific correctness, but they are strong provenance and
integrity controls.


In [13]:
from src.verify_artifacts import (
    verify_models, verify_predictions, verify_metrics_files, verify_figures, make_checksum_file,
)

verify_models(PROJECT_ROOT/"models", cfg["num_folds"])

train_ids = set(train_df["unique_post_id"])
val_ids = set(val_df["unique_post_id"])
test_ids = set(test_df["unique_post_id"])
embargo_ids = set(embargo_df["unique_post_id"])

vf = verify_predictions(oof_pred, val_pred, test_pred, train_ids, val_ids, test_ids, embargo_ids, cfg)
print("prediction verification:", vf)

verify_metrics_files(PROJECT_ROOT/"outputs"/"metrics",
                     ["oof_metrics.json","validation_metrics.json","test_metrics.json"])
verify_figures(PROJECT_ROOT/"outputs"/"figures",
               ["confusion_matrix_validation.png","confusion_matrix_test.png",
                "scatter_validation.png","scatter_test.png"])
print("metrics + figures verified")

files_to_hash = [
    PROJECT_ROOT/"outputs"/"predictions"/"member2_oof_predictions_deterministic.csv",
    PROJECT_ROOT/"outputs"/"predictions"/"member2_validation_predictions_fold_ensemble.csv",
    PROJECT_ROOT/"outputs"/"predictions"/"member2_test_predictions_fold_ensemble.csv",
]
make_checksum_file(PROJECT_ROOT/"outputs"/"logs"/"artifact_checksums.json", files_to_hash)
print("checksum file written")

prediction verification: {'oof_all_train_ids_present': True, 'oof_no_duplicates': np.True_, 'oof_no_val_ids': True, 'oof_no_test_ids': True, 'oof_no_embargo_ids': True, 'oof_rows': 4226, 'val_ids_match': True, 'test_ids_match': True, 'oof_numeric': True, 'oof_within_range': True, 'val_numeric': True, 'val_within_range': True, 'test_numeric': True, 'test_within_range': True, 'val_ensemble_is_mean': True, 'test_ensemble_is_mean': True}
metrics + figures verified


checksum file written


## 12. Tests

### Why have automated tests in a data-science notebook project?

Many serious ML errors are **contract errors**, not syntax errors. A notebook may execute
without exception while using the wrong rows, leaking holdout data, duplicating IDs, or
exporting a schema that Member C cannot safely merge.

The test suite therefore checks invariants such as:

- required input columns and valid target range,
- unique IDs and mutually exclusive modeling roles,
- complete/legal fold assignments,
- OOF coverage and no in-sample prediction leakage,
- validation/test coverage,
- valid prediction ranges,
- Member C handoff columns/schema, and
- expected artifact presence.

The tests complement, rather than replace, scientific evaluation. Passing them means the
pipeline obeys its declared contract; it does **not** by itself mean that the model is
accurate or clinically valid.


In [14]:
import subprocess, sys
test_result = subprocess.run(
    [sys.executable, "-m", "pytest", str(PROJECT_ROOT / "tests"), "-q"],
    capture_output=True, text=True,
)
print(test_result.stdout[-2000:] if len(test_result.stdout) > 0 else "")
print(test_result.stderr[-1000:] if test_result.stderr else "")
assert test_result.returncode == 0, "tests failed"
print("ALL TESTS PASSED")

................                                                         [100%]
16 passed in 4.37s


ALL TESTS PASSED


## 13. Final notebook summary

This final block collects the important reproducibility evidence in one place: model name,
role sizes, fold sizes, prediction coverage, main metrics, artifact paths, and known
limitations.

### How to interpret the final result

A successful run means the Member 2 **pipeline contract** was reproduced: every training
post has exactly one leakage-safe OOF text prediction, every validation/test post has a
five-fold ensemble prediction, and those outputs can be handed to Member C by stable ID.
It does not mean the text model is perfect. In particular, the Very High class has small
support and the text-only model can under-predict the extreme tail. The final fusion is
responsible for combining this text signal with Member 1's complementary structured
information and for the project's final calibrated operating point.

### Known limitations to remember in an oral defense

1. **Label validity:** the stress target is a project annotation, not a clinical diagnosis.
2. **Small high-risk support:** minority-class metrics have higher statistical uncertainty.
3. **Regression-to-the-mean:** extreme true scores can be pulled toward the center.
4. **Token attribution is qualitative:** it is not causal evidence.
5. **Full neural regeneration requires the five fine-tuned fold checkpoints.** If they are
   intentionally stored outside the lightweight ZIP, the frozen handoff predictions still
   allow exact downstream metric/fusion reproduction, while the external checkpoints are
   required for new-post inference.
6. **The test set is evidence, not a tuning set:** no model/threshold change should be made
   because of the locked-test result and then re-reported on the same test as fresh evidence.

### Algorithmic path learned in this notebook

The key methodological lesson is not simply "use BERT." The full reasoning chain is:

**Persian text normalization → pretrained contextual representation → continuous regression
with imbalance/safety-aware loss → grouped five-fold training → OOF stacking predictions →
deterministic holdout ensemble → multi-metric evaluation → post-hoc error analysis →
verified scalar handoff.**

That chain is what makes the Member 2 contribution scientifically defensible and usable by
the final fusion stage.


In [15]:
ok = True

def flag(label, cond):
    global ok
    ok = ok and bool(cond)
    print(("  [OK]  " if cond else "  [FAIL] ") + label)

print("="*70)
print("MEMBER 2 FINAL SUMMARY")
print("="*70)
print("Model:              ", mcfg["model_name"], "(ParsBERT regression, num_labels=1)")
print("Input rows:          train", len(train_df), "| val", len(val_df), "| test", len(test_df), "| embargo", len(embargo_df))
print("Fold sizes:         ", {f: len(ids) for f, ids in folds.items()})
flag("OOF coverage = all train rows, exactly once", set(oof_pred["unique_post_id"])==set(train_df["unique_post_id"]) and oof_pred["unique_post_id"].duplicated().sum()==0)
flag("Validation coverage", set(val_pred["unique_post_id"])==set(val_df["unique_post_id"]))
flag("Test coverage", set(test_pred["unique_post_id"])==set(test_df["unique_post_id"]))
print()
print("OOF MAE:         ", round(oof_rep["MAE"],4), "| Pearson", round(oof_rep["Pearson"],4))
print("Validation MAE:  ", round(val_rep["MAE"],4), "| Pearson", round(val_rep["Pearson"],4), "| acc", round(val_rep["accuracy"],4))
print("Test MAE:        ", round(test_rep["MAE"],4), "| Pearson", round(test_rep["Pearson"],4), "| acc", round(test_rep["accuracy"],4))
print()
print("Artifact paths:")
for p in ["outputs/predictions/member2_oof_predictions_deterministic.csv",
          "outputs/predictions/member2_validation_predictions_fold_ensemble.csv",
          "outputs/predictions/member2_test_predictions_fold_ensemble.csv",
          "outputs/metrics/", "outputs/figures/", "outputs/explainability/"]:
    print("  ", p)
print()
print("Limitations:")
print("  - Very High class is small (25 val / 26 test) and systematically under-predicted.")
print("  - Token attribution is a simple gradient-based approximation, interpretation only.")
print("  - Models require fine-tuned fold checkpoints; base ParsBERT alone is NOT sufficient.")
print("Required for future inference: models/fold_0..4 + tokenizer (or predict.py + model_checkpoint).")
print()
print("="*70)
if ok:
    print("PROJECT REPRODUCTION COMPLETED SUCCESSFULLY")
else:
    print("PROJECT REPRODUCTION FAILED A CHECK — see [FAIL] above")
print("="*70)

MEMBER 2 FINAL SUMMARY
Model:               HooshvareLab/bert-fa-zwnj-base (ParsBERT regression, num_labels=1)
Input rows:          train 4226 | val 452 | test 453 | embargo 484
Fold sizes:          {0: 844, 1: 844, 2: 848, 3: 847, 4: 843}
  [OK]  OOF coverage = all train rows, exactly once
  [OK]  Validation coverage
  [OK]  Test coverage

OOF MAE:          0.9845 | Pearson 0.813
Validation MAE:   0.8246 | Pearson 0.8596 | acc 0.7412
Test MAE:         0.9689 | Pearson 0.8251 | acc 0.7285

Artifact paths:
   outputs/predictions/member2_oof_predictions_deterministic.csv
   outputs/predictions/member2_validation_predictions_fold_ensemble.csv
   outputs/predictions/member2_test_predictions_fold_ensemble.csv
   outputs/metrics/
   outputs/figures/
   outputs/explainability/

Limitations:
  - Very High class is small (25 val / 26 test) and systematically under-predicted.
  - Token attribution is a simple gradient-based approximation, interpretation only.
  - Models require fine-tuned fold c